In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, pearsonr, spearmanr
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

In [ ]:
# Carga inicial del dataset (Asegúrate de que el CSV esté dentro de la carpeta 'data')
RUTA_ARCHIVO = 'data/facturacion_energia.csv'
df_raw = pd.read_csv(RUTA_ARCHIVO, low_memory=False)

print(f"Dimensiones iniciales: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")

# Tabla de perfilamiento
perfil = pd.DataFrame({
    'Variable': df_raw.columns,
    'Tipo_Leido': df_raw.dtypes.astype(str).values,
    'Nulos_Total': df_raw.isnull().sum().values,
    'Porcentaje_Nulos': (df_raw.isnull().mean() * 100).round(3).values,
    'Valores_Distintos': df_raw.nunique().values,
    'Ejemplos': [df_raw[col].dropna().unique()[:4].tolist() for col in df_raw.columns]
})
display(perfil)

In [ ]:
print("=== AUDITORÍA DE ANOMALÍAS EN LOS DATOS ===")
print(f"1. Duplicados (id_cuenta + periodo): {df_raw.duplicated(subset=['id_cuenta', 'periodo']).sum():,}")

for col in ['operador', 'zona', 'estrato', 'tipo_medidor', 'lectura_estimada', 'autogeneracion_solar', 'resultado_inspeccion']:
    print(f"   Únicos en {col}: {df_raw[col].unique()}")
print(f"   Municipios distintos (esperados 26): {df_raw['municipio'].nunique()}")

print(f"3. Consumo < 0: {(df_raw['consumo_kwh'] < 0).sum():,}")
print(f"   Tarifa <= 0: {(df_raw['tarifa_cu_cop_kwh'] <= 0).sum():,}")
print(f"   Días mora < 0: {(df_raw['dias_mora'] < 0).sum():,}")
print(f"   Prepago con mora > 0: {len(df_raw[(df_raw['tipo_medidor'].astype(str).str.lower().str.contains('prepago')) & (df_raw['dias_mora'] > 0)]):,}")

diff_consumo = np.abs(df_raw['valor_consumo_cop'] - (df_raw['consumo_kwh'] * df_raw['tarifa_cu_cop_kwh']))
diff_total = np.abs(df_raw['valor_total_cop'] - (df_raw['valor_consumo_cop'] + df_raw['subsidio_contribucion_cop']))
print(f"5. Error en valor_consumo_cop (> $5): {(diff_consumo > 5).sum():,}")
print(f"   Error en valor_total_cop (> $5): {(diff_total > 5).sum():,}")

In [ ]:
num_cols = ['consumo_kwh', 'tarifa_cu_cop_kwh', 'valor_total_cop', 'dias_mora']
stats_uni = pd.DataFrame({
    'Media': df[num_cols].mean(),
    'Mediana': df[num_cols].median(),
    'Desv_Std': df[num_cols].std(),
    'Asimetria': df[num_cols].skew(),
    'P1': df[num_cols].quantile(0.01),
    'P5': df[num_cols].quantile(0.05),
    'P25': df[num_cols].quantile(0.25),
    'P75': df[num_cols].quantile(0.75),
    'P95': df[num_cols].quantile(0.95),
    'P99': df[num_cols].quantile(0.99),
}).round(2)
display(stats_uni)

In [ ]:
# Esto te muestra la lista limpia de columnas y las primeras filas
print(df.info())
display(df.head(2))

In [ ]:
import pandas as pd

# Cargar el archivo especificando el separador por punto y coma y la ruta correcta
RUTA_ARCHIVO = 'data/facturacion_energia.csv'
df = pd.read_csv(RUTA_ARCHIVO, sep=';', low_memory=False)

# Validar que carguen las filas correctamente
print(f"¡Filas cargadas con éxito: {len(df):,}!")
display(df.head(2))

In [ ]:
import pandas as pd

# Carga rápida solo para ver las columnas reales
df_temp = pd.read_csv('data/facturacion_energia.csv', sep=';', nrows=2)
print("--- LISTA EXACTA DE COLUMNAS EN TU ARCHIVO ---")
for col in df_temp.columns:
    print(f"- '{col}'")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Cargar el dataset sin forzar separador (o con sep=',' si el archivo usa comas)
RUTA_ARCHIVO = 'data/facturacion_energia.csv'
df = pd.read_csv(RUTA_ARCHIVO, low_memory=False)

# Si el archivo usa comas, la línea de arriba leerá las columnas perfecto. 
# Si por alguna razón usa comas pero el texto interno tiene comas, se usaría sep=',' pero aquí vemos que está separado por comas.

# Limpieza básica de espacios por seguridad
df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=True)

print(f"¡Filas cargadas con éxito: {len(df):,}!")
print("Columnas detectadas:", df.columns.tolist())

# 2. Conversión segura de columnas clave a numéricas
df['estrato'] = pd.to_numeric(df['estrato'], errors='coerce')
df['altitud_msnm'] = pd.to_numeric(df['altitud_msnm'], errors='coerce')
df['consumo_kwh'] = pd.to_numeric(df['consumo_kwh'], errors='coerce')

# 3. Configuración y generación de los 4 subplots y la tabla
fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharey='row')
bins_5kwh = np.arange(0, 355, 5)

configs = [
    ((df['estrato'].isin([1, 2, 3])) & (df['altitud_msnm'] < 1000), 173, 'Estratos 1-3 | Límite 173 kWh (<1000 msnm)', axes[0, 0]),
    ((df['estrato'].isin([1, 2, 3])) & (df['altitud_msnm'] >= 1000), 130, 'Estratos 1-3 | Límite 130 kWh (>=1000 msnm)', axes[0, 1]),
    ((df['estrato'].isin([4, 5, 6])) & (df['altitud_msnm'] < 1000), 173, 'Control Estratos 4-6 | Ref. 173 kWh (<1000 msnm)', axes[1, 0]),
    ((df['estrato'].isin([4, 5, 6])) & (df['altitud_msnm'] >= 1000), 130, 'Control Estratos 4-6 | Ref. 130 kWh (>=1000 msnm)', axes[1, 1]),
]

resultados_41 = []

for mask_filtro, lim, titulo, ax in configs:
    sub = df[mask_filtro]['consumo_kwh'].dropna()
    
    print(f"{titulo} -> Registros encontrados: {len(sub):,}")
    
    ax.hist(sub, bins=bins_5kwh, density=True, color='#2b5c8f', alpha=0.8, edgecolor='white')
    ax.axvline(lim, color='red', linestyle='--', linewidth=2, label=f'Límite {lim} kWh')
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_xlim(0, 350)
    ax.legend()
    
    debajo = ((sub >= lim - 10) & (sub < lim)).mean() * 100 if len(sub) > 0 else 0
    encima = ((sub >= lim) & (sub < lim + 10)).mean() * 100 if len(sub) > 0 else 0
    resultados_41.append({
        'Grupo': titulo,
        'Pct_10kWh_Debajo (%)': round(debajo, 2),
        'Pct_10kWh_Encima (%)': round(encima, 2)
    })

plt.tight_layout()
plt.show()
display(pd.DataFrame(resultados_41))

In [ ]:
# 1. Agrupamiento inicial por periodo y operador (como lo tenías)
serie_op = df.groupby(['periodo', 'operador'], observed=True).agg(
    tarifa_mediana=('tarifa_cu_cop_kwh', 'median'),
    consumo_medio=('consumo_kwh', 'mean'),
    prop_mora_60=('dias_mora', lambda x: (x > 60).mean() * 100)
).reset_index()

# Gráfica Tarifa Mediana
plt.figure(figsize=(14, 5))
sns.lineplot(data=serie_op, x='periodo', y='tarifa_mediana', hue='operador', marker='o')
plt.title('Evolución de la Tarifa Mediana Mensual por Operador (2023-2024)', fontweight='bold')
plt.xticks(rotation=45)
plt.ylabel('COP / kWh')
plt.tight_layout()
plt.show()

# Incremento total Ene 2023 vs Dic 2024
t_ini = serie_op[serie_op['periodo'] == '2023-01'].set_index('operador')['tarifa_mediana']
t_fin = serie_op[serie_op['periodo'] == '2024-12'].set_index('operador')['tarifa_mediana']
inc_tarifa = pd.DataFrame({
    'Tarifa_Ene_2023': t_ini,
    'Tarifa_Dic_2024': t_fin,
    'Incremento_Total_%': ((t_fin - t_ini) / t_ini * 100).round(2)
}).sort_values('Incremento_Total_%', ascending=False)
display(inc_tarifa)

# 2. Cambio interanual de consumo (Con agrupamiento previo para evitar duplicados)
serie_op['anio'] = serie_op['periodo'].astype(str).str[:4]
serie_op['mes'] = serie_op['periodo'].astype(str).str[5:7]

# Consolidamos el consumo medio evitando filas duplicadas por mes/operador/año
serie_agrupada = serie_op.groupby(['operador', 'anio', 'mes'], observed=True)['consumo_medio'].mean().reset_index()

# 3. Pivot seguro sin riesgo de ValueError
piv_cons = serie_agrupada.pivot(index=['operador', 'mes'], columns='anio', values='consumo_medio').reset_index()
piv_cons['var_consumo_%'] = (piv_cons['2024'] - piv_cons['2023']) / piv_cons['2023'] * 100
display(piv_cons.groupby('operador', observed=True)['var_consumo_%'].mean().round(2))